#### LangGraph 에이전트 개발 5선 (난이도별)

지난 시간(`exm8.ipynb`)에서 State·Node·Edge·분기·사이클·체크포인터·
human-in-the-loop 같은 LangGraph의 기초 구성요소를 다뤘다. 이번에는 그 구성요소를
조합해서 **실제로 동작하는 에이전트** 5개를 난이도별로 만들어본다.

| 난이도 | 번호 | 제목 | 핵심 개념 |
|---|---|---|---|
| 아주 간단 | 1 | 최소 단일 도구 에이전트 | `create_agent` 한 줄 |
| 아주 간단 | 2 | 수동 ReAct 루프 | `StateGraph` + `ToolNode` + `tools_condition` 직접 구현 |
| 초중급 | 3 | 멀티 도구 라우팅 에이전트 | 도구 3개, 시스템 프롬프트 |
| 초중급 | 4 | Reflection(자기 개선) 에이전트 | 생성→비평→재생성 사이클, 모델 2개 협업 |
| 중급 이상 | 5 | Supervisor 멀티 에이전트 | 서브 에이전트 라우팅 + 체크포인터 + human-in-the-loop |

사용 모델: **gpt-4o-mini**(OpenAI), **exaone3.5**(Ollama, `ollama pull exaone3.5`로 준비)

패키지 버전(2026-07-11 기준 최신 안정판):

```bash
pip install -U langchain==1.3.12 langgraph==1.2.9 \
    langchain-openai==1.3.3 langchain-ollama==1.1.0
```

> 💡 이 노트북의 코드는 실제 API/로컬 LLM 호출을 포함하므로, `OPENAI_API_KEY`
> 환경변수와 로컬 Ollama 서버(`ollama serve`)가 준비된 환경에서 실행해야 한다.

## 레벨 1-1 (아주 간단) — 최소 단일 도구 에이전트

LangChain 1.x의 `create_agent`는 "모델 + 도구 목록"만 주면 **도구 호출 루프를
자동으로 만들어주는** 가장 높은 수준의 진입점이다. 내부적으로는 지난 시간에
배운 StateGraph(LLM 노드 -> 조건부 분기 -> 도구 노드 -> 다시 LLM 노드)가 그대로
돌아간다.

model 파라미터에 `"openai:gpt-4o-mini"`, `"ollama:exaone3.5"` 처럼
**`provider:모델명`** 문자열을 바로 넘길 수 있다 (langchain 1.x부터 지원).

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def calculator(expression: str) -> str:
    """사칙연산 수식 문자열을 계산한다. 예: '3 * (4 + 2)'"""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"계산 오류: {e}"


# gpt-4o-mini 버전
agent_openai = create_agent(model="openai:gpt-4o-mini", tools=[calculator])

# llama3.1 (Ollama) 버전 - 모델만 바꿔도 동일한 방식으로 동작
agent_ollama = create_agent(model="ollama:llama3.1", tools=[calculator])

question = {"messages": [{"role": "user", "content": "37 곱하기 24는 얼마야?"}]}

result_openai = agent_openai.invoke(question)
print("[gpt-4o-mini]", result_openai["messages"][-1].content)

result_ollama = agent_ollama.invoke(question)
print("[llama3.1]  ", result_ollama["messages"][-1].content)

In [ ]:
from typing import TypedDict, List, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.prebuilt import ToolNode
from langchain_core.messages import BaseMessage, HumanMessage

# 1. 상태(State) 정의
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    model_name: str

# 2. 사용할 도구(Tool) 정의
@tool
def calculator(expression: str) -> str:
    """사칙연산 수식 문자열을 계산한다. 예: '3 * (4 + 2)'"""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"계산 오류: {e}"

tools = [calculator]
tool_node = ToolNode(tools) 


# 3. 에이전트 판단 노드
def llm_agent_node(state: AgentState) -> Command:
    # 💡 [안전 장치] ToolNode를 거쳐서 돌아올 때 KeyError가 나는 것을 방지하기 위해 .get()을 사용합니다.
    # 만약 유실되었다면 기본값으로 gpt-4o-mini를 바라보게 설정해두면 안전합니다.
    model_name = state.get("model_name", "gpt-4o-mini")
    
    if model_name.startswith("gpt"):
        llm = ChatOpenAI(model=model_name, temperature=0)
    else:
        llm = ChatOllama(model=model_name, temperature=0)
        
    llm_with_tools = llm.bind_tools(tools)
    response = llm_with_tools.invoke(state["messages"])
    
    if response.tool_calls:
        return Command(
            # 💡 [핵심 변환] model_name이 사라지지 않도록 update 딕셔너리에 함께 누적해 줍니다.
            update={
                "messages": [response],
                "model_name": model_name
            }, 
            goto="tools"  
        )
    else:
        return Command(
            update={
                "messages": [response],
                "model_name": model_name
            },
            goto=END
        )


# 4. 🏗️ 그래프 구조 조립 및 컴파일 (동일)
builder = StateGraph(AgentState)
builder.add_node("llm_agent_node", llm_agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "llm_agent_node")
builder.add_edge("tools", "llm_agent_node")

compiled_agent = builder.compile()


# 5. 🏃‍♂️ 실행 확인
# 💡 [수정] OpenAI가 알아들을 수 있는 정확한 모델명("gpt-4o-mini")으로 변경했습니다.
question = {
    "messages": [HumanMessage(content="37 곱하기 24는 얼마야?")], 
    "model_name": "gpt-4o-mini"  # 👈 로컬 테스트 시 "llama3.1" 등으로 교체 가능!
}
result = compiled_agent.invoke(question)

print("[ToolNode + Command 하이브리드 에이전트 결과]")
print(result["messages"][-1].content)

In [ ]:
# 5. 🏃‍♂️ 실행 확인
# 💡 [수정] OpenAI가 알아들을 수 있는 정확한 모델명("gpt-4o-mini")으로 변경했습니다.
question = {
    "messages": [HumanMessage(content="37 곱하기 24는 얼마야?")], 
    "model_name": "llama3.1"  # 👈 로컬 테스트 시 "llama3.1" 등으로 교체 가능!
}
result = compiled_agent.invoke(question)

print("[ToolNode + Command 하이브리드 에이전트 결과]")
print(result["messages"][-1].content)

## 레벨 1-2 (아주 간단) — 수동 ReAct 루프 직접 구현

`create_agent`가 내부에서 하는 일을 **직접 StateGraph로 조립**해본다. 구조는
"LLM 노드 -> (도구 호출 있으면) 도구 노드 -> 다시 LLM 노드 -> (없으면) 종료"의
사이클이다. `langgraph.prebuilt`가 이 패턴에 필요한 부품(`ToolNode`,
`tools_condition`)을 이미 제공한다.

In [ ]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_ollama import ChatOllama


@tool
def get_current_time(timezone: str = "Asia/Seoul") -> str:
    """현재 시각을 반환한다."""
    from datetime import datetime
    from zoneinfo import ZoneInfo
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S")


tools = [get_current_time]
llm_with_tools = ChatOllama(model="llama3.1", temperature=0).bind_tools(tools)


def call_model(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "call_model")
# tools_condition: 마지막 메시지에 tool_calls가 있으면 "tools", 없으면 END로 라우팅
builder.add_conditional_edges("call_model", tools_condition)
builder.add_edge("tools", "call_model")  # 도구 실행 후 다시 LLM으로 (사이클)

manual_react_agent = builder.compile()

result = manual_react_agent.invoke({"messages": [{"role": "user", "content": "지금 몇 시야?"}]})
for m in result["messages"]:
    print(f"[{type(m).__name__}]", m.content or m.tool_calls)

## 레벨 2-1 (초중급) — 멀티 도구 라우팅 에이전트

도구가 여러 개일 때 LLM이 **질문에 맞는 도구를 스스로 선택**하는 것을 확인한다.
`create_agent`에 도구 목록과 `system_prompt`를 함께 넘긴다. 도구가 여러 개라도
그래프 구조(레벨 1-2에서 본 사이클)는 동일하다 — 달라지는 건 `ToolNode`에
등록된 도구 개수뿐이다.

In [ ]:
from typing import TypedDict, List, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.prebuilt import ToolNode
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

# 1. 상태(State) 정의: 대화 기록이 누적되는 표준 메시지 배열 구조
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

# 2. 제공된 4가지 툴(Tool) 정의 (그대로 유지)
@tool
def calculator(expression: str) -> str:
    """사칙연산 수식 문자열을 계산한다. 예: '3 * (4 + 2)'"""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"계산 오류: {e}"

@tool
def get_current_time(timezone: str = "Asia/Seoul") -> str:
    """현재 시각을 반환한다."""
    from datetime import datetime
    from zoneinfo import ZoneInfo
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S")

@tool
def mock_weather(city: str) -> str:
    """도시 이름으로 현재 날씨를 조회한다 (실습용 가짜 데이터)."""
    data = {"서울": "맑음, 27도", "부산": "흐림, 24도", "제주": "비, 22도"}
    return data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

@tool
def mock_search(query: str) -> str:
    """웹 검색을 수행한다 (실습용 가짜 데이터)."""
    return f"'{query}'에 대한 검색 결과: 관련 문서 3건을 찾았습니다."

# 툴 리스트 및 랭그래프 내장 ToolNode 생성
tools = [calculator, get_current_time, mock_weather, mock_search]
tool_node = ToolNode(tools)  # 기본 노드 이름은 "tools"가 됩니다.


# 3. 에이전트 판단 노드 (gpt-4o-mini 호출 및 라우팅 판단)
def llm_agent_node(state: AgentState) -> Command:
    # 요청하신 대로 gpt-4o-mini 모델로 세팅
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    # 4가지 멀티 툴을 모델에 한 번에 바인딩
    llm_with_tools = llm.bind_tools(tools)
    
    # 모델 호출 (현재 누적된 메시지 전달)
    response = llm_with_tools.invoke(state["messages"])
    
    # ─── 🛠️ OpenAI 네이티브 툴 콜링 기반 자율 라우팅 ───
    if response.tool_calls:
        # 모델이 4가지 툴 중 하나 이상을 호출하겠다고 신호를 주면 "tools" 노드로 이동
        return Command(
            update={"messages": [response]}, 
            goto="tools"  
        )
    else:
        # 툴 호출 없이 일반 텍스트 답변이 나왔다면 최종 답변이므로 종료(END)
        return Command(
            update={"messages": [response]},
            goto=END
        )


# 4. 🏗️ 그래프 구조 조립 및 컴파일
builder = StateGraph(AgentState)

# 노드 등록
builder.add_node("llm_agent_node", llm_agent_node)
builder.add_node("tools", tool_node)

# 정적 엣지 연결
builder.add_edge(START, "llm_agent_node")
builder.add_edge("tools", "llm_agent_node")  # 툴 실행이 끝나면 다시 LLM 판단 노드로 복귀(순환)

routing_agent = builder.compile()


# 5. 🏃‍♂️ 멀티 질문 테스트 실행 확인 파트
questions = [
    "서울 날씨 알려줘",
    "125 나누기 5는?",
    "라그랑주 승수법에 대해 검색해줘",
]

# 기존 시스템 프롬프트 정의
system_prompt = SystemMessage(content="당신은 여러 도구를 상황에 맞게 선택해서 사용하는 한국어 어시스턴트입니다.")

for q in questions:
    # 초기 상태 입력 파트: 시스템 프롬프트와 유저 질문을 묶어서 전달
    initial_input = {
        "messages": [
            system_prompt,
            HumanMessage(content=q)
        ]
    }
    
    # 그래프 실행
    result = routing_agent.invoke(initial_input)
    
    print(f"Q: {q}")
    # 여러 번의 루프(툴 실행 등) 끝에 누적된 전체 메시지 중 최종(마지막) 답변 출력
    print(f"A: {result['messages'][-1].content}\n")

도구가 늘어나도 코드 구조는 그대로다. 다만 **도구 설명(docstring)의 품질이
곧 라우팅 정확도**로 직결된다 — 도구 설명이 모호하면 LLM이 엉뚱한 도구를
고를 수 있으니, 실습에서는 학생들에게 docstring을 명확하게 쓰도록 강조하면 좋다.

## 레벨 2-2 (초중급) — Reflection(자기 개선) 에이전트

"생성 -> 비평 -> 부족하면 재생성"을 반복하는 **사이클** 패턴이다. `exm8.ipynb`에서는
가짜 LLM으로 구조만 확인했지만, 여기서는 **서로 다른 두 모델이 역할을 나눠**
협업한다.

- **생성자(gpt-4o-mini)**: 답변 초안 작성
- **비평자(exaone3.5)**: 한국어로 초안을 평가하고 "PASS" 또는 "REVISE: 이유" 형식으로 응답

비평자가 "PASS"를 내릴 때까지, 또는 최대 횟수까지 반복한다.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
critic_llm = ChatOllama(model="llama3.1", temperature=0)

MAX_ROUNDS = 3


class ReflectionState(TypedDict):
    topic: str
    draft: str
    feedback: str
    round: int


def generate_node(state: ReflectionState) -> dict:
    prompt = f"주제: {state['topic']}\n요청: 3문장 이내로 핵심만 설명해줘."
    if state["feedback"]:
        prompt += f"\n이전 피드백을 반영해서 다시 작성해줘: {state['feedback']}"
    draft = generator_llm.invoke(prompt).content
    return {"draft": draft, "round": state["round"] + 1}


def critique_node(state: ReflectionState) -> dict:
    prompt = (
        f"다음 글이 3문장 이내로 핵심을 잘 설명하는지 평가해줘.\n\n{state['draft']}\n\n"
        "충분하면 'PASS'만 출력하고, 부족하면 'REVISE: 이유'형식으로 한 줄로 답해줘."
    )
    feedback = critic_llm.invoke(prompt).content.strip()
    return {"feedback": feedback}


def should_continue(state: ReflectionState) -> str:
    if state["feedback"].upper().startswith("PASS") or state["round"] >= MAX_ROUNDS:
        return "done"
    return "retry"


builder = StateGraph(ReflectionState)
builder.add_node("generate", generate_node)
builder.add_node("critique", critique_node)

builder.add_edge(START, "generate")
builder.add_edge("generate", "critique")
builder.add_conditional_edges("critique", should_continue, {"retry": "generate", "done": END})

reflection_agent = builder.compile()

result = reflection_agent.invoke({"topic": "LangGraph의 체크포인터", "draft": "", "feedback": "", "round": 0})
print(f"최종 초안 ({result['round']}회차):\n{result['draft']}")
print(f"\n최종 피드백: {result['feedback']}")

> 💡 생성자와 비평자를 **서로 다른 모델**로 분리하면, 비평자가 생성자의 표현을
> 그대로 답습하는 "자기 확신 편향"을 줄일 수 있다는 장점이 있다 (실무에서는
> 비평자로 더 저렴하거나 특성이 다른 모델을 쓰는 경우가 많다).

## 레벨 3 (중급 이상) — Supervisor 멀티 에이전트 + 체크포인터 + Human-in-the-loop

지금까지 배운 요소를 모두 조합한다.

- **Supervisor 노드**: 사용자 요청을 보고 다음에 어떤 서브 에이전트를 부를지 결정
- **researcher 서브 에이전트**: `mock_search` 도구를 쓰는 `create_agent` (gpt-4o-mini)
- **writer 서브 에이전트**: 조사 결과를 한국어 보고서로 정리하는 `create_agent` (exaone3.5)
- **Human-in-the-loop**: 최종 보고서를 확정하기 전에 사람 승인을 받음 (`interrupt`)
- **체크포인터**: 승인 대기 상태를 저장해뒀다가, 사람이 응답하면 그 지점부터 재개

Supervisor -> researcher -> writer -> (사람 승인) -> 종료 순서로 흐르는 그래프다.

In [ ]:
from typing import Annotated, Literal, TypedDict
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage


# --- 서브 에이전트 두 개 (레벨 2-1에서 본 create_agent 패턴 재사용) ---
researcher = create_agent(
    model="openai:gpt-4o-mini",
    tools=[mock_search],
    system_prompt="당신은 자료 조사 담당자입니다. mock_search로 필요한 정보를 찾아 요약하세요.",
)
writer = create_agent(
    model="ollama:llama3.1",
    tools=[],
    system_prompt="당신은 조사 내용을 바탕으로 한국어 보고서를 3문장 이내로 작성하는 작가입니다.",
)


class SupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    research_result: str
    report: str
    approved: bool


def supervisor_node(state: SupervisorState) -> Command[Literal["researcher", "writer", "await_approval"]]:
    """단순 규칙 기반 라우팅: research_result가 없으면 조사부터, 있는데 report가 없으면 작성으로."""
    if not state["research_result"]:
        return Command(goto="researcher")
    if not state["report"]:
        return Command(goto="writer")
    return Command(goto="await_approval")


def researcher_node(state: SupervisorState) -> dict:
    topic = state["messages"][-1].content
    result = researcher.invoke({"messages": [HumanMessage(topic)]})
    return {"research_result": result["messages"][-1].content}


def writer_node(state: SupervisorState) -> dict:
    prompt = f"조사 내용: {state['research_result']}\n이 내용을 바탕으로 보고서를 작성해줘."
    result = writer.invoke({"messages": [HumanMessage(prompt)]})
    return {"report": result["messages"][-1].content}


def await_approval_node(state: SupervisorState) -> dict:
    decision = interrupt({"report": state["report"], "question": "이 보고서를 승인하시겠습니까? (yes/no)"})
    return {"approved": decision == "yes"}


builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)
builder.add_node("await_approval", await_approval_node)

builder.add_edge(START, "supervisor")
builder.add_edge("researcher", "supervisor")   # 조사 후 다시 supervisor가 다음 단계 결정
builder.add_edge("writer", "supervisor")       # 작성 후 다시 supervisor가 다음 단계 결정
builder.add_edge("await_approval", END)

supervisor_graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "report-session-1"}}

# 1) 최초 실행 -> researcher -> writer -> await_approval에서 interrupt로 정지
state1 = supervisor_graph.invoke(
    {"messages": [HumanMessage("LangGraph 체크포인터 기능을 조사해서 보고서를 써줘")],
     "research_result": "", "report": "", "approved": False},
    config=config,
)
print("정지 시점의 보고서 초안:\n", state1["report"])

# 2) 사람이 승인("yes")했다고 가정하고 재개
state2 = supervisor_graph.invoke(Command(resume="yes"), config=config)
print("\n최종 승인 여부:", state2["approved"])

> 💡 `supervisor_node`가 `Command(goto=...)`를 직접 반환하는 방식은
> `add_conditional_edges`를 쓰는 것과 동등하지만, **다음 노드로 갈 때 상태 갱신과
> 라우팅을 한 번에 처리**할 수 있어서 멀티 에이전트처럼 노드가 많아질 때
> 코드가 더 깔끔해진다. 실무에서 서브 에이전트를 3개 이상으로 늘릴 때는
> `supervisor_node`의 규칙을 LLM 기반 분류기로 바꾸면 그대로 확장된다.

## 정리

| 레벨 | 예제 | 배운 것 |
|---|---|---|
| 아주 간단 | 1. 최소 단일 도구 에이전트 | `create_agent`로 도구 호출 루프 자동 생성 |
| 아주 간단 | 2. 수동 ReAct 루프 | `ToolNode` + `tools_condition`으로 루프 직접 조립 |
| 초중급 | 3. 멀티 도구 라우팅 | 도구가 늘어나도 그래프 구조는 동일 |
| 초중급 | 4. Reflection 에이전트 | 두 모델 협업 + 품질 기반 사이클 |
| 중급 이상 | 5. Supervisor 멀티 에이전트 | 서브 에이전트 조합 + 체크포인터 + human-in-the-loop |
